# ResNet: Deep Residual Learning for Image Recognition

## Learning Objectives
1. Understand the degradation problem and why residual connections solve it
2. Implement basic and bottleneck residual blocks from scratch
3. Train a small ResNet on CIFAR-10 with residual connections
4. Compare ResNet with pre-trained models and fine-tuning strategies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")

## Level 1: Basic Residual Block

The core idea: instead of learning H(x) directly, learn F(x) = H(x) - x (the residual)

In [ ]:
class BasicResidualBlock(nn.Module):
    """Simplest residual block: Conv → BN → ReLU → Conv → BN + skip connection"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection (identity or projection when dimensions change)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual  # Key: element-wise addition with skip connection
        out = self.relu(out)
        return out

# Test on synthetic data
batch_size, channels, height, width = 4, 3, 32, 32
x = torch.randn(batch_size, channels, height, width, device=device)
block = BasicResidualBlock(in_channels=3, out_channels=64, stride=1).to(device)

output = block(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Block parameters: {sum(p.numel() for p in block.parameters()):,}")

## Level 2: Small ResNet-20 on Synthetic Data

Full network with multiple residual blocks, demonstrating training dynamics

In [ ]:
class ResNet(nn.Module):
    """ResNet: Stack of residual blocks"""
    
    def __init__(self, block_class, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 16
        
        # Initial convolution
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        
        # Residual layers with increasing channels and stride
        self.layer1 = self._make_layer(block_class, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block_class, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block_class, 64, num_blocks[2], stride=2)
        
        # Global average pooling and classification head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)
    
    def _make_layer(self, block_class, channels, blocks, stride):
        layers = []
        # First block in layer may have stride and channel change
        layers.append(block_class(self.in_channels, channels, stride=stride))
        self.in_channels = channels
        # Remaining blocks
        for _ in range(1, blocks):
            layers.append(block_class(channels, channels, stride=1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# ResNet-20: [3, 3, 3] blocks per layer = 9 residual blocks total
model = ResNet(BasicResidualBlock, [3, 3, 3], num_classes=10).to(device)
print(f"ResNet-20 parameters: {sum(p.numel() for p in model.parameters()):,}")

# Create synthetic dataset
train_size, test_size = 500, 100
X_train = torch.randn(train_size, 3, 32, 32)
y_train = torch.randint(0, 10, (train_size,))
X_test = torch.randn(test_size, 3, 32, 32)
y_test = torch.randint(0, 10, (test_size,))

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# Training loop
train_losses = []
test_accs = []

for epoch in range(5):
    model.train()
    epoch_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    scheduler.step()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    train_losses.append(avg_loss)
    test_accs.append(acc)
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Test Acc={acc:.2f}%")

print("\nResNet-20 training complete!")

## Real-World Example 1: Bottleneck Blocks (ResNet-50 style)

ResNet-50+ uses bottleneck blocks (1x1 reduce -> 3x3 -> 1x1 expand) for parameter efficiency

In [ ]:
class BottleneckBlock(nn.Module):
    """Bottleneck block: 1×1 (reduce) → 3×3 → 1×1 (expand), with skip connection"""
    expansion = 4  # Output channels = 4 × internal channels
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # 1×1 reduce to out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 3×3 convolution at reduced channel size
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1×1 expand to out_channels * expansion
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion,
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        
        # Skip connection (projection if dimensions change)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )
    
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = out + residual
        out = self.relu(out)
        return out

# Compare parameters and FLOPs
basic_block = BasicResidualBlock(64, 64)
bottleneck_block = BottleneckBlock(64, 64)

basic_params = sum(p.numel() for p in basic_block.parameters())
bottleneck_params = sum(p.numel() for p in bottleneck_block.parameters())

print(f"Basic Block:")
print(f"  Parameters: {basic_params:,}")
print(f"  Structure: 3×3 → 3×3")
print()
print(f"Bottleneck Block:")
print(f"  Parameters: {bottleneck_params:,}")
print(f"  Structure: 1×1 reduce → 3×3 → 1×1 expand")
print()
print(f"Ratio (Bottleneck / Basic): {bottleneck_params / basic_params:.2f}x")
print(f"\nBottleneck is better for deep networks: reduces FLOPs by ~4x")

## Real-World Example 2: Analyzing Gradient Flow

Demonstrate why residual connections help gradients flow through deep networks

In [ ]:
# Create a simple plain network for comparison
class PlainBlock(nn.Module):
    """Plain block without skip connection"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return out

# Stack multiple blocks and measure gradient magnitudes
def measure_gradient_flow(block_class, num_blocks, block_name):
    """Measure how gradient magnitudes change through deep stacks"""
    x = torch.randn(1, 3, 32, 32, requires_grad=True)
    
    blocks = []
    for i in range(num_blocks):
        if i == 0:
            blocks.append(block_class(3, 64))
        else:
            blocks.append(block_class(64, 64))
    
    blocks = nn.Sequential(*blocks)
    
    # Forward pass
    output = blocks(x)
    loss = output.sum()
    loss.backward()
    
    # Measure input gradient magnitude
    input_grad = x.grad.abs().mean().item()
    return input_grad

# Measure for different depths
depths = [2, 5, 10, 20]
residual_grads = []
plain_grads = []

for depth in depths:
    res_grad = measure_gradient_flow(BasicResidualBlock, depth, "Residual")
    plain_grad = measure_gradient_flow(PlainBlock, depth, "Plain")
    residual_grads.append(res_grad)
    plain_grads.append(plain_grad)

print(f"Gradient Flow Analysis (input gradient magnitude):")
print(f"Depth | Residual Block | Plain Block | Ratio")
print(f"-" * 50)
for d, rg, pg in zip(depths, residual_grads, plain_grads):
    ratio = rg / (pg + 1e-8)
    print(f"{d:2d}    | {rg:.6f}      | {pg:.6f}     | {ratio:.2f}x")

print(f"\nKey observation: Residual blocks maintain stronger gradients through deep networks!")

## Real-World Example 3: Transfer Learning Strategy

Fine-tuning strategy for using pre-trained ResNet on custom tasks

In [ ]:
# Simulate transfer learning strategy
class TransferLearningStrategy:
    """Different strategies for fine-tuning pre-trained ResNets"""
    
    @staticmethod
    def strategy_1_frozen_backbone(model, num_classes=10):
        """Freeze all layers except final classification head"""
        # Freeze backbone
        for param in model.parameters():
            param.requires_grad = False
        
        # Unfreeze only FC layer
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
        for param in model.fc.parameters():
            param.requires_grad = True
        
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        return trainable, total
    
    @staticmethod
    def strategy_2_partial_finetune(model, num_classes=10, num_unfreeze=2):
        """Freeze early layers, fine-tune last num_unfreeze layers"""
        # Freeze all
        for param in model.parameters():
            param.requires_grad = False
        
        # Unfreeze last num_unfreeze layers and FC
        layers_to_unfreeze = list(model.children())[-num_unfreeze:]
        for layer in layers_to_unfreeze:
            for param in layer.parameters():
                param.requires_grad = True
        
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
        for param in model.fc.parameters():
            param.requires_grad = True
        
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        return trainable, total
    
    @staticmethod
    def strategy_3_full_finetune(model, num_classes=10, lr_ratio=0.1):
        """Fine-tune all layers with discriminative learning rates"""
        # Unfreeze all
        for param in model.parameters():
            param.requires_grad = True
        
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
        
        trainable = sum(p.numel() for p in model.parameters())
        total = trainable
        return trainable, total

# Create model for demonstration
model_demo = ResNet(BasicResidualBlock, [3, 3, 3], num_classes=10).to(device)
total_params = sum(p.numel() for p in model_demo.parameters())

print(f"Transfer Learning Strategies for ResNet-20 ({total_params:,} total params):")
print()

# Strategy 1
model1 = ResNet(BasicResidualBlock, [3, 3, 3], num_classes=10).to(device)
trainable1, total1 = TransferLearningStrategy.strategy_1_frozen_backbone(model1)
print(f"Strategy 1: Frozen Backbone")
print(f"  Trainable: {trainable1:,} ({100*trainable1/total1:.1f}%)")
print(f"  Best for: Quick domain adaptation with small datasets")
print()

# Strategy 2
model2 = ResNet(BasicResidualBlock, [3, 3, 3], num_classes=10).to(device)
trainable2, total2 = TransferLearningStrategy.strategy_2_partial_finetune(model2, num_unfreeze=2)
print(f"Strategy 2: Partial Fine-tune (last 2 layers)")
print(f"  Trainable: {trainable2:,} ({100*trainable2/total2:.1f}%)")
print(f"  Best for: Balanced approach with medium datasets")
print()

# Strategy 3
model3 = ResNet(BasicResidualBlock, [3, 3, 3], num_classes=10).to(device)
trainable3, total3 = TransferLearningStrategy.strategy_3_full_finetune(model3)
print(f"Strategy 3: Full Fine-tune (all layers)")
print(f"  Trainable: {trainable3:,} ({100*trainable3/total3:.1f}%)")
print(f"  Best for: Large datasets with significant distribution shift")
print()

print(f"\nKey principle: Use lower learning rates for earlier layers")
print(f"(they contain general visual features, require less adaptation)")

## Key Takeaways

**Core idea:** Skip connections solve the degradation problem by enabling direct gradient flow

**Block variants:**
| Type | Structure | Use Case |
|------|-----------|----------|
| Basic | 3×3 → 3×3 + skip | ResNet-18/34, smaller models |
| Bottleneck | 1×1 → 3×3 → 1×1 + skip | ResNet-50+, efficiency-critical |

**When to use ResNet:**
- Depth > 30 layers (ResNet-50 for most tasks)
- Transfer learning from ImageNet
- Need strong gradient flow through deep networks

**Related papers:**
- [Vision Transformer](./02-vision-transformer.md) - Modern alternative
- [LoRA](../nlp/concepts/05-lora.md) - Efficient fine-tuning

In [ ]:
# Visualization: Training convergence comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Loss convergence
axes[0].plot(train_losses, marker='o', linewidth=2, label='ResNet-20 Training Loss')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=11)
axes[0].set_title('ResNet-20 Convergence on Synthetic Data', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Test accuracy
axes[1].plot(test_accs, marker='s', linewidth=2, color='green', label='Test Accuracy')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_title('ResNet-20 Test Accuracy', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
axes[1].set_ylim([0, 100])

plt.tight_layout()
plt.savefig('/tmp/resnet_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

print("ResNet implementation complete!")
print(f"\nSummary:")
print(f"  - Skip connections enable training of very deep networks")
print(f"  - Bottleneck blocks reduce FLOPs while maintaining capacity")
print(f"  - Transfer learning: use pre-trained ResNet-50 for most tasks")
print(f"  - Fine-tuning strategy depends on dataset size and domain similarity")